In [ ]:

from pathlib import Path
import os

from pydantic_settings import BaseSettings


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    os.getenv(
        "PROJECT_ROOT",
        str(Path(__file__).resolve().parents[2]),
    )
)

BACKEND_DIR = PROJECT_ROOT / "backend"

ENV_FILE = BACKEND_DIR / ".env"


# ============================================================
# EXPLICIT .ENV LOADER
# ============================================================

def load_env_file(path: Path) -> None:
    """
    Load backend/.env into os.environ.

    Existing environment variables are preserved.
    Secrets are never printed.
    """

    if not path.exists():
        raise FileNotFoundError(
            f"Environment file not found: {path}"
        )

    for raw_line in path.read_text(
        encoding="utf-8"
    ).splitlines():

        line = raw_line.strip()

        if (
            not line
            or line.startswith("#")
            or "=" not in line
        ):
            continue

        key, value = line.split("=", 1)

        key = key.strip()
        value = value.strip()

        # Remove optional quotes
        if (
            len(value) >= 2
            and value[0] == value[-1]
            and value[0] in ("'", '"')
        ):
            value = value[1:-1]

        # Do not overwrite existing environment variables
        if key and key not in os.environ:
            os.environ[key] = value


if ENV_FILE.exists():
    load_env_file(ENV_FILE)


# ============================================================
# SETTINGS
# ============================================================

class Settings(BaseSettings):

    # ========================================================
    # Application
    # ========================================================

    app_name: str = "AI Study Companion"
    app_env: str = "development"
    debug: bool = True

    # ========================================================
    # MongoDB
    # ========================================================

    mongodb_uri: str = ""
    mongodb_database: str = "ai_study_companion"

    # ========================================================
    # JWT
    # ========================================================

    jwt_secret_key: str = ""
    jwt_algorithm: str = "HS256"
    jwt_expire_minutes: int = 60

    # ========================================================
    # GROQ
    # ========================================================

    groq_model: str = "openai/gpt-oss-120b"

    groq_api_key_1: str | None = None
    groq_api_key_2: str | None = None
    groq_api_key_3: str | None = None
    groq_api_key_4: str | None = None
    groq_api_key_5: str | None = None
    groq_api_key_6: str | None = None

    groq_max_key_attempts: int = 6

    # ========================================================
    # LOCAL EMBEDDINGS
    # ========================================================

    embedding_model: str = (
        "sentence-transformers/all-MiniLM-L6-v2"
    )

    embedding_dimensions: int = 384

    # ========================================================
    # DIRECTORIES
    # ========================================================

    upload_dir: str = "data/uploads"
    processed_dir: str = "data/processed"


# ============================================================
# CREATE SETTINGS FROM ENVIRONMENT
# ============================================================

settings = Settings(
    app_name=os.getenv(
        "APP_NAME",
        "AI Study Companion",
    ),

    app_env=os.getenv(
        "APP_ENV",
        "development",
    ),

    debug=os.getenv(
        "DEBUG",
        "true",
    ).lower() == "true",

    mongodb_uri=os.getenv(
        "MONGODB_URI",
        "",
    ),

    mongodb_database=os.getenv(
        "MONGODB_DATABASE",
        "ai_study_companion",
    ),

    jwt_secret_key=os.getenv(
        "JWT_SECRET_KEY",
        "",
    ),

    jwt_algorithm=os.getenv(
        "JWT_ALGORITHM",
        "HS256",
    ),

    jwt_expire_minutes=int(
        os.getenv(
            "JWT_EXPIRE_MINUTES",
            "60",
        )
    ),

    groq_model=os.getenv(
        "GROQ_MODEL",
        "openai/gpt-oss-120b",
    ),

    groq_api_key_1=os.getenv(
        "GROQ_API_KEY_1"
    ),

    groq_api_key_2=os.getenv(
        "GROQ_API_KEY_2"
    ),

    groq_api_key_3=os.getenv(
        "GROQ_API_KEY_3"
    ),

    groq_api_key_4=os.getenv(
        "GROQ_API_KEY_4"
    ),

    groq_api_key_5=os.getenv(
        "GROQ_API_KEY_5"
    ),

    groq_api_key_6=os.getenv(
        "GROQ_API_KEY_6"
    ),

    groq_max_key_attempts=int(
        os.getenv(
            "GROQ_MAX_KEY_ATTEMPTS",
            "6",
        )
    ),

    embedding_model=os.getenv(
        "EMBEDDING_MODEL",
        "sentence-transformers/all-MiniLM-L6-v2",
    ),

    embedding_dimensions=int(
        os.getenv(
            "EMBEDDING_DIMENSIONS",
            "384",
        )
    ),

    upload_dir=os.getenv(
        "UPLOAD_DIR",
        "data/uploads",
    ),

    processed_dir=os.getenv(
        "PROCESSED_DIR",
        "data/processed",
    ),
)


# ============================================================
# VALIDATION
# ============================================================

if not settings.mongodb_uri:
    raise RuntimeError(
        "MONGODB_URI is not configured."
    )

if not any(
    [
        settings.groq_api_key_1,
        settings.groq_api_key_2,
        settings.groq_api_key_3,
        settings.groq_api_key_4,
        settings.groq_api_key_5,
        settings.groq_api_key_6,
    ]
):
    raise RuntimeError(
        "No GROQ_API_KEY_1 ... GROQ_API_KEY_6 are configured."
    )

if settings.embedding_dimensions != 384:
    raise RuntimeError(
        "Embedding dimensions must be 384."
    )
